# Data Pipeline Notebook

This notebook combines:
1. Raw data collection (Fundamentals, Stock Prices, Options)
2. Point-in-time adjustment
3. Feature engineering
4. Integrity and coverage validation


## 1) Environment And Imports


In [2]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Ensure project-root imports when running notebook from notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "team_t").exists():
    TEAM_ROOT = CWD / "team_t"
elif CWD.name == "team_t":
    TEAM_ROOT = CWD
elif CWD.name == "notebooks" and CWD.parent.name == "team_t":
    TEAM_ROOT = CWD.parent
else:
    TEAM_ROOT = next((p for p in [CWD, *CWD.parents] if p.name == "team_t"), None)

if TEAM_ROOT is None:
    raise FileNotFoundError("Could not resolve team_t directory from current working directory")

if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from src.data_utils import (
    configure_api_from_env,
    fetch_zacks_table,
    load_prices_csv_required,
    build_static_top10_universe,
    prepare_fundamentals_with_availability,
    asof_join_point_in_time,
    validate_point_in_time_panel,
    connect_wrds,
    pull_optionmetrics_calls_atm_dataset,
)
from src.feature_engineering import (
    add_split_adjusted_intraday_prices,
    add_fundamental_change_features,
    add_price_liquidity_features,
    add_staged_features,
    get_stage_feature_columns,
    compute_price_to_book,
    compute_rolling_beta_vs_spy,
    winsorize_cross_sectional,
    zscore_cross_sectional,
)


## 2) Parameters And Output Paths


In [3]:
# Full-history defaults for forward-walk workflows.
PRICE_START = pd.Timestamp("1900-01-01")
PRICE_END = pd.Timestamp.today().normalize()
RANK_DATE = pd.Timestamp("2012-12-31")
LAG_DAYS = 45

BETA_WINDOW = 252
BETA_MIN_OBS = 126
FEATURE_STAGE_MAX = 5

OPT_START = None
OPT_END = None

# Cache controls for faster reruns.
# Set any of these to True when you need a fresh pull/recompute.
USE_CACHE = True
REFRESH_FUNDAMENTALS = False
REFRESH_PRICES = False
REFRESH_OPTIONS = False
REFRESH_FEATURE_PANEL = False

PRICES_CSV = TEAM_ROOT / "data" / "PRICES.csv"
OUTPUT_DIR = TEAM_ROOT / "data" / "lstm_draft" / "processed"
OPTIONS_DIR = TEAM_ROOT / "data" / "options"
for path in [OUTPUT_DIR, OPTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

UNIVERSE_CSV = OUTPUT_DIR / "lstm_static_top10_universe_2012_12_31.csv"

# Raw exports (full-history canonical)
PRICES_ADJ_PARQUET_FULL = OUTPUT_DIR / "prices_adjusted_full_history.parquet"
FUNDAMENTALS_PARQUET_FULL = OUTPUT_DIR / "fundamentals_with_availability_full_history.parquet"
BETA_PARQUET_FULL = OUTPUT_DIR / "beta_252d_full_history.parquet"
OPTIONS_PARQUET_FULL = OPTIONS_DIR / "optionmetrics_calls_atm_20_60d_full_history.parquet"

# Raw exports (legacy names)
PRICES_ADJ_PARQUET = OUTPUT_DIR / "prices_adjusted_2006_2013.parquet"
FUNDAMENTALS_PARQUET = OUTPUT_DIR / "fundamentals_with_availability_2006_2013.parquet"
BETA_PARQUET = OUTPUT_DIR / "beta_252d_2006_2013.parquet"
OPTIONS_PARQUET = OPTIONS_DIR / "optionmetrics_calls_atm_20_60d_2006_2013.parquet"

# Feature panel exports
FEATURE_PANEL_PARQUET = OUTPUT_DIR / "feature_panel_pit_normalized_full_history.parquet"
FEATURE_PANEL_CSV = OUTPUT_DIR / "feature_panel_pit_normalized_full_history.csv"

# Backward-compatible panel names consumed in older flows.
PANEL_PARQUET_FULL = OUTPUT_DIR / "lstm_feature_panel_full_history.parquet"
PANEL_CSV_FULL = OUTPUT_DIR / "lstm_feature_panel_full_history.csv"
PANEL_PARQUET = OUTPUT_DIR / "lstm_feature_panel_2006_2013.parquet"
PANEL_CSV = OUTPUT_DIR / "lstm_feature_panel_2006_2013.csv"


## 3) Credentials Setup


In [4]:
env_candidates = [
    TEAM_ROOT / ".env",
    TEAM_ROOT.parent / ".env",
    Path.cwd() / ".env",
]
configure_api_from_env(env_candidates)


## Fundamentals (quarterly)

Used to construct the signal.

Examples:
- ROE
- profit_margin
- debt_equity
- EPS
- book_value_per_share

These come from Zacks or similar sources.


In [5]:
mt_cols = ["ticker", "sp500_member_flag"]
mktv_cols = ["ticker", "per_end_date", "per_type", "mkt_val"]
fr_cols = [
    "ticker", "per_end_date", "per_type",
    "tot_debt_tot_equity", "ret_equity", "profit_margin", "book_val_per_share",
]
fc_cols = ["ticker", "per_end_date", "per_type", "diluted_net_eps"]

if (
    USE_CACHE
    and not REFRESH_FUNDAMENTALS
    and UNIVERSE_CSV.exists()
    and FUNDAMENTALS_PARQUET_FULL.exists()
):
    top10 = pd.read_csv(UNIVERSE_CSV)
    fundamentals = pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
    static_tickers = top10["ticker_price"].astype(str).tolist()
    if "ticker" in top10.columns:
        raw_tickers = sorted(top10["ticker"].astype(str).tolist())
    else:
        raw_tickers = sorted(top10["ticker_price"].astype(str).tolist())

    if "per_end_date" in fundamentals.columns:
        fundamentals["per_end_date"] = pd.to_datetime(fundamentals["per_end_date"], errors="coerce")
    if "feature_available_date" in fundamentals.columns:
        fundamentals["feature_available_date"] = pd.to_datetime(
            fundamentals["feature_available_date"], errors="coerce"
        )

    print(f"[cache hit] fundamentals={FUNDAMENTALS_PARQUET_FULL}")
    print(f"[cache hit] universe={UNIVERSE_CSV}")
else:
    mt = fetch_zacks_table("ZACKS/MT", mt_cols, filters={"sp500_member_flag": "Y"}, paginate=True)
    mktv = fetch_zacks_table(
        "ZACKS/MKTV",
        mktv_cols,
        filters={"per_end_date": str(RANK_DATE.date())},
        paginate=True,
    )

    top10 = build_static_top10_universe(mt_df=mt, mktv_df=mktv, rank_date=str(RANK_DATE.date()))
    static_tickers = top10["ticker_price"].tolist()
    raw_tickers = sorted(top10["ticker"].tolist())

    date_lo = PRICE_START.date().isoformat()
    date_hi = PRICE_END.date().isoformat()

    fr = fetch_zacks_table(
        "ZACKS/FR",
        fr_cols,
        filters={"ticker": {"in": raw_tickers}, "per_end_date": {"between": (date_lo, date_hi)}},
        paginate=True,
    )
    fc = fetch_zacks_table(
        "ZACKS/FC",
        fc_cols,
        filters={"ticker": {"in": raw_tickers}, "per_end_date": {"between": (date_lo, date_hi)}},
        paginate=True,
    )
    fundamentals = prepare_fundamentals_with_availability(fr, fc, lag_days=LAG_DAYS)

print(f"[universe] tickers={len(static_tickers)} sample={static_tickers[:5]}")
print(f"[fundamentals] rows={len(fundamentals):,} tickers={fundamentals['ticker_price'].nunique():,}")
print(f"[fundamentals] per_end_date min={fundamentals['per_end_date'].min()} max={fundamentals['per_end_date'].max()}")


[cache hit] fundamentals=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/fundamentals_with_availability_full_history.parquet
[cache hit] universe=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_static_top10_universe_2012_12_31.csv
[universe] tickers=10 sample=['AAPL', 'XOM', 'GOOGL', 'WMT', 'MSFT']
[fundamentals] rows=801 tickers=10
[fundamentals] per_end_date min=2006-01-31 00:00:00 max=2026-01-31 00:00:00


## Stock Prices (daily)

Required for:
- return targets
- delta hedge
- realized volatility

Fields:
- date
- ticker
- open
- close
- adj_open
- adj_close
- volume


In [6]:
price_cols = ["ticker", "date", "open", "close", "adj_close", "volume"]
price_tickers = sorted(set(static_tickers + ["SPY"]))

if USE_CACHE and not REFRESH_PRICES and PRICES_ADJ_PARQUET_FULL.exists():
    prices_adj = pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
    prices_adj["date"] = pd.to_datetime(prices_adj["date"], errors="coerce")
    print(f"[cache hit] prices_adj={PRICES_ADJ_PARQUET_FULL}")
else:
    prices = load_prices_csv_required(
        csv_path=PRICES_CSV,
        tickers=price_tickers,
        start=PRICE_START.date().isoformat(),
        end=PRICE_END.date().isoformat(),
        usecols=price_cols,
    )
    prices_adj = add_split_adjusted_intraday_prices(prices)

if prices_adj.empty:
    raise ValueError("No price rows returned for selected ticker universe and date range.")

print(f"[prices_adj] rows={len(prices_adj):,} tickers={prices_adj['ticker'].nunique():,}")
print(f"[prices_adj] date_min={prices_adj['date'].min()} date_max={prices_adj['date'].max()}")
prices_adj[["date", "ticker", "open", "close", "adj_open", "adj_close", "volume"]].head(3)


[cache hit] prices_adj=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/prices_adjusted_full_history.parquet
[prices_adj] rows=127,368 tickers=11
[prices_adj] date_min=1962-01-02 00:00:00 date_max=2024-11-05 00:00:00


,date,ticker,open,close,adj_open,adj_close,volume
0,1980-12-12,AAPL,28.75,28.75,0.099192,0.099192,2093900.0
1,1980-12-15,AAPL,27.38,27.25,0.094466,0.094017,785200.0
2,1980-12-16,AAPL,25.37,25.25,0.087531,0.087117,472000.0


### Export Raw Fundamentals + Prices


In [7]:
top10.to_csv(UNIVERSE_CSV, index=False)
fundamentals.to_parquet(FUNDAMENTALS_PARQUET_FULL, index=False)
prices_adj.to_parquet(PRICES_ADJ_PARQUET_FULL, index=False)

# Backward-compatible raw filenames
fundamentals.to_parquet(FUNDAMENTALS_PARQUET, index=False)
prices_adj.to_parquet(PRICES_ADJ_PARQUET, index=False)

print(f"[export] universe={UNIVERSE_CSV}")
print(f"[export] fundamentals_full={FUNDAMENTALS_PARQUET_FULL}")
print(f"[export] prices_adj_full={PRICES_ADJ_PARQUET_FULL}")


[export] universe=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_static_top10_universe_2012_12_31.csv
[export] fundamentals_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/fundamentals_with_availability_full_history.parquet
[export] prices_adj_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/prices_adjusted_full_history.parquet


## Options Data (daily)

From OptionMetrics IvyDB US.

Used for:
- ATM option prices
- delta
- implied volatility

Example fields:
- date
- ticker
- strike
- expiration
- delta
- mid_price
- implied_vol
- open_interest


In [8]:
opt_start_date = OPT_START if OPT_START is not None else prices_adj["date"].min().date().isoformat()
print(f"[options pull window] start={opt_start_date} end={OPT_END if OPT_END is not None else 'latest available'}")

use_options_cache = USE_CACHE and not REFRESH_OPTIONS and OPTIONS_PARQUET_FULL.exists()
options_df = None

if use_options_cache:
    cached = pd.read_parquet(OPTIONS_PARQUET_FULL)
    dte_ok = ("dte" in cached.columns) and cached["dte"].between(30, 60).all()
    cp_ok = ("cp_flag" in cached.columns) and cached["cp_flag"].astype(str).str.upper().eq("C").all()
    if dte_ok and cp_ok:
        options_df = cached
        print(f"[cache hit] options={OPTIONS_PARQUET_FULL}")
        print(f"[cache hit] rows={len(options_df):,} tickers={options_df['ticker'].nunique():,}")
    else:
        print("[cache stale] options cache does not satisfy current rules (calls-only, DTE 30-60). Re-pulling.")

if options_df is None:
    db = connect_wrds()
    try:
        options_df = pull_optionmetrics_calls_atm_dataset(
            db=db,
            universe_path=UNIVERSE_CSV,
            fallback_universe_path=UNIVERSE_CSV,
            output_path=OPTIONS_PARQUET_FULL,
            start_date=opt_start_date,
            end_date=OPT_END,
        )
    finally:
        db.close()
        print("[db] WRDS connection closed")

if OPTIONS_PARQUET_FULL.exists() and OPTIONS_PARQUET_FULL != OPTIONS_PARQUET:
    options_df.to_parquet(OPTIONS_PARQUET, index=False)

print(f"[options] rows={len(options_df):,} tickers={options_df['ticker'].nunique():,}")
print(f"[options] output_full={OPTIONS_PARQUET_FULL}")
print(f"[options] output_legacy={OPTIONS_PARQUET}")


[options pull window] start=1962-01-02 end=latest available
[cache hit] options=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_full_history.parquet
[cache hit] rows=62,971 tickers=9
[options] rows=62,971 tickers=9
[options] output_full=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_full_history.parquet
[options] output_legacy=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_2006_2013.parquet


## Point-In-Time Adjustment + Feature Engineering

This section:
- aligns fundamentals by availability date using PIT as-of join
- computes change-based and price/liquidity features
- applies cross-sectional winsorization and z-score normalization


In [9]:
expected_stage_features = get_stage_feature_columns(max_stage=FEATURE_STAGE_MAX)

use_feature_cache = (
    USE_CACHE
    and not REFRESH_FEATURE_PANEL
    and FEATURE_PANEL_PARQUET.exists()
    and BETA_PARQUET_FULL.exists()
)

if use_feature_cache:
    panel_out = pd.read_parquet(FEATURE_PANEL_PARQUET)
    beta_df = pd.read_parquet(BETA_PARQUET_FULL)
    panel_out["date"] = pd.to_datetime(panel_out["date"], errors="coerce")

    feature_cols = [col for col in expected_stage_features if col in panel_out.columns]
    missing_stage_cols = [col for col in expected_stage_features if col not in panel_out.columns]

    if missing_stage_cols:
        print(
            "[cache stale] feature panel missing staged columns: "
            f"{missing_stage_cols}. Recomputing feature panel."
        )
        use_feature_cache = False
    else:
        print(f"[cache hit] feature_panel={FEATURE_PANEL_PARQUET}")
        print(f"[cache hit] feature_stage={FEATURE_STAGE_MAX} feature_count={len(feature_cols)}")

if not use_feature_cache:
    fund_pti = fundamentals.copy()
    fund_pti["ticker"] = fund_pti["ticker_price"].astype(str).str.upper().str.strip()

    prices_model = prices_adj[prices_adj["ticker"].isin(static_tickers)].copy()
    prices_model["ticker"] = prices_model["ticker"].astype(str).str.upper().str.strip()
    panel = asof_join_point_in_time(
        prices_df=prices_model,
        fundamentals_df=fund_pti,
        on_date_col="date",
        by_ticker_col="ticker",
    )
    validate_point_in_time_panel(panel)

    panel_feat = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    panel_feat = compute_price_to_book(panel_feat)

    # Stage 1 baseline features (unchanged formulas).
    panel_feat = add_fundamental_change_features(panel_feat, ticker_col="ticker", date_col="date")
    panel_feat = add_price_liquidity_features(
        panel_feat,
        ticker_col="ticker",
        date_col="date",
        price_col="adj_close",
        volume_col="volume",
        volume_window=20,
        min_volume_obs=5,
    )

    beta_df = compute_rolling_beta_vs_spy(
        prices_df=prices_adj[["ticker", "date", "adj_close"]].copy(),
        window=BETA_WINDOW,
        min_obs=BETA_MIN_OBS,
    )
    panel_feat = panel_feat.merge(beta_df[["ticker", "date", "beta_252d"]], on=["ticker", "date"], how="left")
    panel_feat = panel_feat.rename(columns={"beta_252d": "rolling_beta"})

    # Stages 2-5 derived features are cumulative and use availability-time alignment.
    panel_feat = add_staged_features(
        panel_df=panel_feat,
        max_stage=FEATURE_STAGE_MAX,
        ticker_col="ticker",
        date_col="date",
        feature_available_col="feature_available_date",
        price_col="adj_close",
        reaction_clip=(-0.05, 0.05),
        vol_epsilon=1e-8,
    )

    feature_cols = [col for col in expected_stage_features if col in panel_feat.columns]
    missing_stage_cols = [col for col in expected_stage_features if col not in panel_feat.columns]
    if missing_stage_cols:
        raise KeyError(f"Missing staged feature columns after build: {missing_stage_cols}")

    # One-pass normalization after all staged raw features are generated.
    panel_feat = winsorize_cross_sectional(
        panel_df=panel_feat,
        feature_cols=feature_cols,
        date_col="date",
        lower_q=0.01,
        upper_q=0.99,
    )
    panel_feat = zscore_cross_sectional(
        panel_df=panel_feat,
        feature_cols=feature_cols,
        date_col="date",
        suffix="_z",
    )

    # Keep backward-compatible next-day target alongside engineered features.
    panel_feat["adj_close"] = pd.to_numeric(panel_feat["adj_close"], errors="coerce")
    panel_feat.loc[panel_feat["adj_close"] <= 0, "adj_close"] = np.nan
    panel_feat["target_next_log_ret"] = panel_feat.groupby("ticker")["adj_close"].transform(
        lambda s: np.log(s.shift(-1)) - np.log(s)
    )

    panel_out = panel_feat.sort_values(["ticker", "date"]).reset_index(drop=True)

panel_out = panel_out.sort_values(["ticker", "date"]).reset_index(drop=True)
duplicates = int(panel_out.duplicated(["ticker", "date"]).sum())
assert duplicates == 0, f"Found {duplicates} duplicate (ticker, date) rows in panel_out"

# Export engineered feature panel.
panel_out.to_parquet(FEATURE_PANEL_PARQUET, index=False)
panel_out.to_csv(FEATURE_PANEL_CSV, index=False)

# Backward-compatible panel exports.
panel_out.to_parquet(PANEL_PARQUET_FULL, index=False)
panel_out.to_csv(PANEL_CSV_FULL, index=False)
panel_out.to_parquet(PANEL_PARQUET, index=False)
panel_out.to_csv(PANEL_CSV, index=False)

# Export beta artifacts retained from previous raw-data workflow.
beta_df.to_parquet(BETA_PARQUET_FULL, index=False)
beta_df.to_parquet(BETA_PARQUET, index=False)

print(f"[panel] rows={len(panel_out):,} tickers={panel_out['ticker'].nunique():,}")
print(f"[panel] date_min={panel_out['date'].min()} date_max={panel_out['date'].max()}")
print(f"[panel] feature_stage={FEATURE_STAGE_MAX}")
print(f"[panel] feature_cols={feature_cols}")
print(f"[export] feature_panel_parquet={FEATURE_PANEL_PARQUET}")
print(f"[export] feature_panel_csv={FEATURE_PANEL_CSV}")
print(f"[export] beta_full={BETA_PARQUET_FULL}")


[cache stale] feature panel missing staged columns: ['roe_change_accel', 'margin_change_accel', 'eps_growth_surprise', 'reaction_speed', 'vol_regime_ratio_20_60']. Recomputing feature panel.
[panel] rows=119,368 tickers=10
[panel] date_min=1962-01-02 00:00:00 date_max=2024-11-05 00:00:00
[panel] feature_stage=5
[panel] feature_cols=['roe_change', 'margin_change', 'leverage_change', 'eps_growth', 'book_value_growth', 'log_return', 'rolling_beta', 'volume_ratio', 'roe_change_accel', 'margin_change_accel', 'eps_growth_surprise', 'reaction_speed', 'vol_regime_ratio_20_60']
[export] feature_panel_parquet=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/feature_panel_pit_normalized_full_history.parquet
[export] feature_panel_csv=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/feature_panel_pit_normalized_full_history.csv
[export] beta_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/beta_252d_full_history.parquet


## 4) Integrity And Summary Of Data


In [10]:
def _date_min_max(df: pd.DataFrame, date_col: str) -> tuple[str | None, str | None]:
    if df is None or df.empty or date_col not in df.columns:
        return None, None
    d = pd.to_datetime(df[date_col], errors="coerce")
    return (
        str(d.min().date()) if d.notna().any() else None,
        str(d.max().date()) if d.notna().any() else None,
    )

_fund_df = fundamentals if "fundamentals" in globals() else pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
_prices_df = prices_adj if "prices_adj" in globals() else pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
_opt_df = options_df if "options_df" in globals() else pd.read_parquet(OPTIONS_PARQUET_FULL)
_panel_df = panel_out if "panel_out" in globals() else pd.read_parquet(FEATURE_PANEL_PARQUET)

coverage = pd.DataFrame(
    [
        {
            "dataset": "Fundamentals (quarterly)",
            "date_field": "per_end_date",
            "date_min": _date_min_max(_fund_df, "per_end_date")[0],
            "date_max": _date_min_max(_fund_df, "per_end_date")[1],
            "rows": int(len(_fund_df)),
            "tickers": int(_fund_df["ticker_price"].nunique()) if "ticker_price" in _fund_df.columns else None,
        },
        {
            "dataset": "Fundamentals availability (PIT)",
            "date_field": "feature_available_date",
            "date_min": _date_min_max(_fund_df, "feature_available_date")[0],
            "date_max": _date_min_max(_fund_df, "feature_available_date")[1],
            "rows": int(len(_fund_df)),
            "tickers": int(_fund_df["ticker_price"].nunique()) if "ticker_price" in _fund_df.columns else None,
        },
        {
            "dataset": "Stock Prices (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_prices_df, "date")[0],
            "date_max": _date_min_max(_prices_df, "date")[1],
            "rows": int(len(_prices_df)),
            "tickers": int(_prices_df["ticker"].nunique()) if "ticker" in _prices_df.columns else None,
        },
        {
            "dataset": "Options Data (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_opt_df, "date")[0],
            "date_max": _date_min_max(_opt_df, "date")[1],
            "rows": int(len(_opt_df)),
            "tickers": int(_opt_df["ticker"].nunique()) if "ticker" in _opt_df.columns else None,
        },
        {
            "dataset": "PIT Feature Panel (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_panel_df, "date")[0],
            "date_max": _date_min_max(_panel_df, "date")[1],
            "rows": int(len(_panel_df)),
            "tickers": int(_panel_df["ticker"].nunique()) if "ticker" in _panel_df.columns else None,
        },
    ]
)
coverage


,dataset,date_field,date_min,date_max,rows,tickers
0,Fundamentals (quarterly),per_end_date,2006-01-31,2026-01-31,801,10
1,Fundamentals availability (PIT),feature_available_date,2006-03-17,2026-03-17,801,10
2,Stock Prices (daily),date,1962-01-02,2024-11-05,127368,11
3,Options Data (daily),date,1996-01-04,2025-08-29,62971,9
4,PIT Feature Panel (daily),date,1962-01-02,2024-11-05,119368,10


In [11]:
# Validate required raw-data blocks and expected fields.
_fund_df = fundamentals if "fundamentals" in globals() else pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
_prices_df = prices_adj if "prices_adj" in globals() else pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
_opt_df = options_df if "options_df" in globals() else pd.read_parquet(OPTIONS_PARQUET_FULL)
_panel_df = panel_out if "panel_out" in globals() else pd.read_parquet(FEATURE_PANEL_PARQUET)

required_fund = {
    "ROE": ["ret_equity", "roe"],
    "profit_margin": ["profit_margin"],
    "debt_equity": ["tot_debt_tot_equity", "debt_equity"],
    "EPS": ["diluted_net_eps", "eps", "eps_basic"],
    "book_value_per_share": ["book_val_per_share", "book_value_per_share"],
}
required_prices = {
    "date": ["date"],
    "ticker": ["ticker"],
    "open": ["open"],
    "close": ["close"],
    "adj_open": ["adj_open"],
    "adj_close": ["adj_close", "adj_close_intraday"],
    "volume": ["volume"],
}
required_options = {
    "date": ["date"],
    "ticker": ["ticker"],
    "strike": ["strike_price", "strike"],
    "expiration": ["exdate", "expiration"],
    "delta": ["delta"],
    "mid_price": ["mid_price"],
    "implied_vol": ["implied_vol", "implied_volatility", "impl_volatility"],
    "open_interest": ["open_interest"],
}
required_panel = {
    "date": ["date"],
    "ticker": ["ticker"],
}
required_panel.update({col: [col] for col in get_stage_feature_columns(max_stage=FEATURE_STAGE_MAX)})

def _present(requirements: dict[str, list[str]], cols: list[str]) -> dict[str, str | None]:
    colset = set(cols)
    out: dict[str, str | None] = {}
    for logical, aliases in requirements.items():
        out[logical] = next((a for a in aliases if a in colset), None)
    return out

def _status(mapping: dict[str, str | None]) -> str:
    return "PASS" if all(v is not None for v in mapping.values()) else "FAIL"

rows = [
    {
        "dataset": "Fundamentals (quarterly)",
        "status": _status(_present(required_fund, list(_fund_df.columns))),
    },
    {
        "dataset": "Stock Prices (daily)",
        "status": _status(_present(required_prices, list(_prices_df.columns))),
    },
    {
        "dataset": "Options Data (daily)",
        "status": _status(_present(required_options, list(_opt_df.columns))),
    },
    {
        "dataset": "PIT Feature Panel (daily)",
        "status": _status(_present(required_panel, list(_panel_df.columns))),
    },
]
validation_table = pd.DataFrame(rows)

if (validation_table["status"] == "FAIL").any():
    raise AssertionError("Validation failed:\n" + validation_table.to_string(index=False))

validation_table


,dataset,status
0,Fundamentals (quarterly),PASS
1,Stock Prices (daily),PASS
2,Options Data (daily),PASS
3,PIT Feature Panel (daily),PASS


In [13]:
# Final artifact integrity check.
required_files = [
    UNIVERSE_CSV,
    FUNDAMENTALS_PARQUET_FULL,
    PRICES_ADJ_PARQUET_FULL,
    BETA_PARQUET_FULL,
    OPTIONS_PARQUET_FULL,
    FEATURE_PANEL_PARQUET,
    FEATURE_PANEL_CSV,
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required exported files: {missing_files}")

opt_chk = pd.read_parquet(OPTIONS_PARQUET_FULL)
required_option_cols = {
    "date", "ticker", "secid", "cp_flag", "strike_price", "exdate", "dte",
    "best_bid", "best_offer", "mid_price", "implied_vol", "delta",
    "underlying_price", "moneyness", "option_mid_return",
}
missing_option_cols = sorted(required_option_cols - set(opt_chk.columns))
if missing_option_cols:
    raise KeyError(f"Options missing columns: {missing_option_cols}")

checks = {}
if not opt_chk.empty:
    checks["calls_only"] = bool(opt_chk["cp_flag"].astype(str).str.upper().eq("C").all())
    checks["dte_30_60"] = bool(opt_chk["dte"].between(30, 60).all())
    checks["positive_bid"] = bool((opt_chk["best_bid"] > 0).all())
    checks["positive_offer"] = bool((opt_chk["best_offer"] > 0).all())
    checks["moneyness_band"] = bool(opt_chk["moneyness"].between(0.95, 1.05).all())
    checks["one_contract_per_ticker_date"] = bool(not opt_chk.duplicated(["ticker", "date"]).any())
    if "open_interest" in opt_chk.columns and opt_chk["open_interest"].notna().any():
        checks["positive_open_interest"] = bool((opt_chk["open_interest"] > 0).all())

integrity_manifest = {
    "fundamentals_rows": int(len(pd.read_parquet(FUNDAMENTALS_PARQUET_FULL))),
    "prices_rows": int(len(pd.read_parquet(PRICES_ADJ_PARQUET_FULL))),
    "options_rows": int(len(opt_chk)),
    "feature_panel_rows": int(len(pd.read_parquet(FEATURE_PANEL_PARQUET))),
    "checks": checks,
}
integrity_manifest


{'fundamentals_rows': 801,
 'prices_rows': 127368,
 'options_rows': 62971,
 'feature_panel_rows': 119368,
 'checks': {'calls_only': True,
  'dte_30_60': True,
  'positive_bid': True,
  'positive_offer': True,
  'moneyness_band': True,
  'one_contract_per_ticker_date': True,
  'positive_open_interest': True}}